In [3]:
# 2 by 2 grid world, initial state values
S = [0, 0, 0, 0]
policy = [0, 0, 0, 0]
q_values = [[0, 0, 0, 0] for _ in range(4)]
k = 5

def state_update(state, action):
    # state transition function
    if action == 0:  # up
        if state in [2, 3]:
            return state - 2
        return state
    if action == 1:  # down
        if state in [0, 1]:
            return state + 2
        return state
    if action == 2:  # left
        if state in [1, 3]:
            return state - 1    
        else:
            return state
    if action == 3:  # right
        if state in [0, 2]:
            return state + 1
        else:
            return state

for i in range(k):
    for s in range(4):
        for a in range(4):
            next_state = state_update(s, a)
            reward = 1 if next_state == 3 else 0
            q_values[s][a] = reward + 0.9 * max(q_values[next_state])
            ## update policy 
            policy[s] = q_values[s].index(max(q_values[s]))
            ## update the state value
            S[s] = max(q_values[s])
            

# Extract the optimal policy
# for s in range(4):
#     policy[s] = q_values[s].index(max(q_values[s]))
    
# Print the optimal policy
print("Optimal Policy:")
for s in range(4):
    print(f"State {s}: Action {policy[s]}")

# Print the state values
print("\nState Values:")
for s in range(4):
    print(f"State {s}: Value {S[s]}")
    
# Print the Q-values
print("\nQ-values:")
for s in range(4):
    print(f"State {s}: {q_values[s]}")

Optimal Policy:
State 0: Action 1
State 1: Action 1
State 2: Action 3
State 3: Action 3

State Values:
State 0: Value 5.3433240080229005
State 1: Value 6.708991607220611
State 2: Value 6.708991607220611
State 3: Value 7.03809244649855

Q-values:
State 0: [3.9839835501000005, 5.3433240080229005, 4.8089916072206105, 5.3433240080229005]
State 1: [5.3433240080229005, 6.708991607220611, 4.8089916072206105, 6.03809244649855]
State 2: [4.8089916072206105, 5.3433240080229005, 5.3433240080229005, 6.708991607220611]
State 3: [6.03809244649855, 6.708991607220611, 6.03809244649855, 7.03809244649855]


In [10]:
class GridWorldValueIteration:
    def __init__(self, width, height, rewards=None, gamma=0.9):
        self.width = width
        self.height = height
        self.n_tiles = width * height
        self.rewards = list(rewards) if rewards is not None else [0.0] * self.n_tiles
        if len(self.rewards) != self.n_tiles:
            raise ValueError("rewards must have length width * height")
        self.gamma = gamma
        self.n_actions = 4
        self.action_symbols = {0: '↑', 1: '↓', 2: '←', 3: '→'}
        self.q_values = [[0.0] * self.n_actions for _ in range(self.n_tiles)]
        self.policy = [0] * self.n_tiles
        self.values = [0.0] * self.n_tiles

    def to_index(self, row, col):
        return row * self.width + col

    def to_coord(self, state):
        return divmod(state, self.width)

    def next_state(self, state, action):
        row, col = self.to_coord(state)
        if action == 0:  # up
            row = max(0, row - 1)
        elif action == 1:  # down
            row = min(self.height - 1, row + 1)
        elif action == 2:  # left
            col = max(0, col - 1)
        elif action == 3:  # right
            col = min(self.width - 1, col + 1)
        return self.to_index(row, col)

    def update(self, n_iterations=50):
        for _ in range(n_iterations):
            for state in range(self.n_tiles):
                for action in range(self.n_actions):
                    ns = self.next_state(state, action)
                    reward = self.rewards[ns]
                    self.q_values[state][action] = reward + self.gamma * max(self.q_values[ns])
                best_action = self.q_values[state].index(max(self.q_values[state]))
                self.policy[state] = best_action
                self.values[state] = max(self.q_values[state])

    def policy_grid(self):
        grid = []
        for row in range(self.height):
            grid.append([self.action_symbols[self.policy[self.to_index(row, col)]] for col in range(self.width)])
        return grid

    def values_grid(self):
        grid = []
        for row in range(self.height):
            grid.append([round(self.values[self.to_index(row, col)], 2) for col in range(self.width)])
        return grid

    def print_policy(self):
        print("Learned policy:")
        for row in self.policy_grid():
            print(" ".join(row))
        print("\nState values:")
        for row in self.values_grid():
            print(" ".join(f"{v:.2f}" for v in row))

# Example usage:
rewards = [0.0, 0.0, -1.0, 0.0,
           -1.0, 0.0, 0.0, -1.0,
           0.0, 0.0, 0.0, 0.0,
           -1.0, 0.0, 0.0, 1.0]
agent = GridWorldValueIteration(width=4, height=4, rewards=rewards, gamma=0.9)
agent.update(n_iterations=10)
agent.print_policy()

Learned policy:
→ ↓ ↓ ↓
↓ ↓ ↓ ↓
→ ↓ ↓ ↓
→ → → →

State values:
3.96 4.81 5.72 5.68
4.81 5.72 6.68 7.72
5.72 6.68 7.72 8.85
6.68 7.72 8.85 8.97
